# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousefelshazly/FlyRankMachineLearning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
So One row = one content item on one report date for one client, where gsc_data_available IS TRUE. We focus on GSC columns only. GA4 columns are excluded because only 4.2% of rows have available GA4 data — not enough for reliable modeling.

In [ ]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

# Read token safely from Colab Secrets — never paste in the cell
token = userdata.get("HF_TOKEN")

# Connect to DuckDB (runs in memory, no installation needed)
con = duckdb.connect()

# Give DuckDB our HuggingFace credentials
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

# Point to the warehouse base path
rel = "hf://datasets/FlyRank/internship-warehouse"

# Quick sanity check — count rows in March 2026 only
# We filter by the partition folder path (month=2026-03)
result = con.sql(f"""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as earliest,
           MAX(report_date) as latest
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   earliest     latest
0    9841378 2026-03-01 2026-03-31


In [ ]:
# Peek at one row — .T flips it vertically so long rows are readable
sample = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 1
""").df()

print(sample.T)

                                                 0
report_date                    2026-03-01 00:00:00
client_hash_id             client_73cda7b4e4f265ea
content_hash_id           content_b7e512995f79d5a6
client_has_gsc                                True
client_has_ga4                               False
gsc_data_available                            True
ga4_data_available                            <NA>
gsc_impressions                                 20
gsc_clicks                                       0
gsc_sum_position                                67
gsc_avg_position                              3.35
ga4_pageviews                                 <NA>
ga4_sessions                                  <NA>
ga4_users                                     <NA>
ga4_engaged_sessions                          <NA>
ga4_total_engagement_sec                      <NA>
sessions_organic                              <NA>
sessions_direct                               <NA>
sessions_referral              

In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) as gsc_pct,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) as ga4_pct
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(availability) #results show 4% have ga4

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available  ga4_available  gsc_pct  ga4_pct
0     9841378      3611061.0       413966.0     36.7      4.2


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

gsc_impressions — search visibility, knowable at decision time
gsc_clicks — search engagement, knowable at decision time
gsc_avg_position — raw daily ranking, knowable at decision time

Label:

is_declining (derived) — did this page's average position worsen over the following 30 days compared to the previous 30 days? Binary: 1 = declining, 0 = stable or improving

Context (never fed to model):

report_date — used for time window calculations
client_hash_id — ID for grouping and splitting
content_hash_id — ID for identifying pages

Excluded:

gsc_sum_position — redundant, avg_position already captures this
client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — filter flags, not model signals
All GA4 columns — only 4.2% of rows have real GA4 data, excluding to preserve 95% of dataset

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(grain_check)
print(f"Duplicate rows found: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
Duplicate rows found: 0


In [ ]:
availability_check = con.sql(f"""
    SELECT COUNT(*) as rows_with_gsc
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

print(availability_check)

   rows_with_gsc
0        3611061


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print (result)

   row_count   earliest     latest
0    9841378 2026-03-01 2026-03-31


In [ ]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,

        -- Feature 1: average daily impressions
        AVG(gsc_impressions) as avg_impressions_30d,

        -- Feature 2: click through rate
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) as ctr,

        -- Feature 3: average position
        AVG(gsc_avg_position) as avg_position_30d,

        -- Feature 4: impression trend (second half vs first half of month)
        AVG(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions END) -
        AVG(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions END) as impression_trend,

        -- Feature 5: click trend (second half vs first half of month)
        AVG(CASE WHEN report_date >= '2026-03-16' THEN gsc_clicks END) -
        AVG(CASE WHEN report_date < '2026-03-16' THEN gsc_clicks END) as click_trend

    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

print(f"Feature frame shape: {features.shape}")
print(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)
            content_hash_id           client_hash_id  avg_impressions_30d  \
0  content_ac8663da7484669a  client_62f4a7e64f5e0096             2.000000   
1  content_39d7361b4945d504  client_62f4a7e64f5e0096             3.208333   
2  content_d49a012dcb924e31  client_62f4a7e64f5e0096            10.612903   
3  content_614baf2af4330bd7  client_62f4a7e64f5e0096            24.903226   
4  content_225dc9235023be5f  client_62f4a7e64f5e0096            15.741935   

      ctr  avg_position_30d  impression_trend  click_trend  
0  0.0000          4.909314         -0.472222     0.000000  
1  0.0000          4.074107         -1.577778     0.000000  
2  0.0000          5.177774        -11.212500     0.000000  
3  0.0013          4.685335         -5.095833    -0.066667  
4  0.0020         17.148172         -5.537500    -0.066667  


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# STEP 1 — honest model (impression_trend excluded — it's the label source)
features['label'] = (features['impression_trend'] < 0).astype(int)

honest_cols = ['avg_impressions_30d', 'ctr', 'avg_position_30d', 'click_trend']
X_honest = features[honest_cols].fillna(0)
y = features['label']

clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_honest, y)
honest_score = accuracy_score(y, clf.predict(X_honest))
print(f"Honest score: {honest_score:.3f}")

# STEP 2 — add the leaky column (impression_trend derives the label)
X_leaky = features[honest_cols + ['impression_trend']].fillna(0)
clf.fit(X_leaky, y)
leaky_score = accuracy_score(y, clf.predict(X_leaky))
print(f"Leaky score:  {leaky_score:.3f}")
print(f"Score jump:   +{leaky_score - honest_score:.3f} ← this is the leak")

# STEP 3 — delete the leak, keep honest score
print(f"\nLeaky column removed. Final honest score: {honest_score:.3f}")

Honest score: 0.988
Leaky score:  1.000
Score jump:   +0.012 ← this is the leak

Leaky column removed. Final honest score: 0.988


In [ ]:
print(features['label'].value_counts())
print(features['label'].value_counts(normalize=True))

label
0    111280
1     65458
Name: count, dtype: int64
label
0    0.629633
1    0.370367
Name: proportion, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limitation:

GSC data only captures search visibility — impressions, clicks, and ranking position. It cannot tell us what happens after a user enters the page. A page may appear to be stable in search rankings while user engagement is silently deteriorating — high bounce rates, low scroll depth, poor session quality. These signals live in GA4, which was excluded due to only 4.2% row availability. Therefore our declining label is based purely on position trends and may miss pages that are declining in quality but not yet in ranking

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.